# 02 — Complaint Classification

**Objective:** Apply a keyword-based multi-label classifier to assign each complaint one or more issue categories, then resolve a single `primary_category` for downstream aggregation and SQL analysis.

**Why this matters:** FCC complaints arrive as unstructured text with no built-in issue taxonomy. Without categorization, volume trend analysis is blind to *what* is driving complaints — you can see that Georgia had 289 complaints but not that billing issues account for 73% of them. Classification is the step that makes complaint data actionable for a CX or operations team.

**Approach:** Rule-based keyword matching (`src/classify.py`). Each complaint can match multiple categories; `primary_category` is resolved via a priority ordering weighted toward more specific, high-signal categories (billing, contract) over generic ones (customer service).

**Known limitation:** Keyword overlap across categories is high. The median `category_count` in this dataset is ~5–6 out of 9 possible labels, which suggests over-triggering rather than genuine multi-issue complexity. `category_count` should be read as a rough complexity proxy, not a precise count. A supervised classifier trained on labeled data would produce cleaner results.

**Input:** `data/processed/cleaned_comcast_complaints.csv`  
**Output:** `data/processed/cleaned_comcast_complaints.csv` (with classification columns added)

In [ ]:
import sys
import pandas as pd

# Allow importing from src/ when running notebook from notebooks/
sys.path.append("..")

from src.classify import classify_complaint_multi, get_primary_category, CATEGORY_PRIORITY

CLEANED_PATH = "../data/processed/cleaned_comcast_complaints.csv"
CLASSIFIED_PATH = "../data/processed/classified_comcast_complaints.csv"

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv(CLEANED_PATH, parse_dates=["date"])
print(f"Loaded {len(df)} rows")
df[["ticket_id", "complaint_text", "state", "status"]].head(3)

## 2. Apply Multi-Label Classification

In [ ]:
df["complaint_categories"] = df["complaint_text"].apply(classify_complaint_multi)
df["primary_category"] = df["complaint_categories"].apply(get_primary_category)
df["category_count"] = df["complaint_categories"].apply(len)

df[["complaint_text", "complaint_categories", "primary_category", "category_count"]].head(5)

## 3. Validate Classification Results

In [ ]:
# Primary category distribution
primary_dist = df["primary_category"].value_counts()
primary_pct = df["primary_category"].value_counts(normalize=True).mul(100).round(1)

pd.DataFrame({"count": primary_dist, "pct": primary_pct})

In [ ]:
# Category count distribution — high median (5-6) indicates keyword overlap, not genuine complexity
print("category_count distribution:")
print(df["category_count"].value_counts().sort_index())
print(f"\nMedian category_count: {df['category_count'].median()}")

In [ ]:
# Spot-check: review a sample of complaints classified as each primary category
# Useful for sanity-checking whether keyword matching is hitting the right topics
for cat in ["billing_issue", "network_issue", "outage_compensation", "other"]:
    sample = df[df["primary_category"] == cat]["complaint_text"].sample(
        min(2, (df["primary_category"] == cat).sum()), random_state=42
    )
    print(f"\n--- {cat} ---")
    for text in sample:
        print(f"  {text[:200]}...")

## 4. Flag Low-Confidence Classifications

Complaints classified as `other` (no keyword matches) and single-category complaints (category_count = 1) are the most reliable signal. Flag `other` for manual review.

In [ ]:
unclassified = df[df["primary_category"] == "other"]
print(f"Unclassified complaints ('other'): {len(unclassified)} ({len(unclassified)/len(df)*100:.1f}%)")
unclassified[["ticket_id", "complaint_text"]].head(5)

## 5. Save Classified Dataset

In [ ]:
df.to_csv(CLASSIFIED_PATH, index=False)
print(f"Saved classified dataset to {CLASSIFIED_PATH}")
print(f"Columns: {df.columns.tolist()}")